In [ ]:
from sysdata.sim.csv_futures_sim_data import csvFuturesSimData

data = csvFuturesSimData()

from systems.provided.rules.ewmac import ewmac_forecast_with_defaults as ewmac

from systems.forecasting import Rules


Configuring sim logging


In [2]:
data.get_instrument_list()

my_rules = Rules(ewmac)
my_rules.trading_rules()


{'rule0': TradingRule; function: <function ewmac_forecast_with_defaults at 0x130d2f1c0>, data: data.daily_prices (args: {}) and other_args: }

In [ ]:
my_rules = Rules(dict(ewmac=ewmac))
my_rules.trading_rules()


{'ewmac': TradingRule; function: <function ewmac_forecast_with_defaults at 0x130d2f1c0>, data: data.daily_prices (args: {}) and other_args: }

In [ ]:
from systems.basesystem import System

my_system = System([my_rules], data)
my_system

my_system.rules.get_raw_forecast("SP500", "ewmac")


2026-08-27 12:33:17 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-08-27 12:33:17 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-08-27 12:33:17 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-08-27 12:33:17 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SP500'} Calculating raw forecast SP500 for ewmac


index
1982-09-14         NaN
1982-09-15         NaN
1982-09-16         NaN
1982-09-17         NaN
1982-09-20         NaN
                ...   
2024-03-22    8.360741
2024-03-25    8.563601
2024-03-26    8.760027
2024-03-27    8.842813
2024-03-28    9.096310
Freq: B, Name: price, Length: 10838, dtype: float64

In [ ]:
from systems.trading_rules import TradingRule

ewmac_8 = TradingRule((ewmac, [], dict(Lfast=8, Lslow=32)))
ewmac_32 = TradingRule(dict(function=ewmac, other_args=dict(Lfast=32, Lslow=128)))

my_rules = Rules(dict(ewmac8=ewmac_8, ewmac32=ewmac_32))
my_rules.trading_rules()["ewmac32"]


TradingRule; function: <function ewmac_forecast_with_defaults at 0x130d2f1c0>, data: data.daily_prices (args: {}) and other_args: Lfast, Lslow

In [ ]:
my_system = System([my_rules], data)
my_system.rules.get_raw_forecast("SP500", "ewmac32").tail(5)


2026-08-27 16:07:29 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-08-27 16:07:29 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-08-27 16:07:29 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-08-27 16:07:29 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SP500'} Calculating raw forecast SP500 for ewmac32


index
2024-03-22    8.360741
2024-03-25    8.563601
2024-03-26    8.760027
2024-03-27    8.842813
2024-03-28    9.096310
Freq: B, Name: price, dtype: float64

In [ ]:
from sysdata.config.configdata import Config

my_config = Config()
my_config


Config with elements: 

In [ ]:
empty_rules = Rules()
my_config.trading_rules = dict(ewmac8=ewmac_8, ewmac32=ewmac_32)
my_system = System([empty_rules], data, config=my_config)
my_system.rules.get_raw_forecast("SP500", "ewmac32").tail(5)


2026-08-27 16:28:06 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-08-27 16:28:06 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-08-27 16:28:06 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-08-27 16:28:06 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SP500'} Calculating raw forecast SP500 for ewmac32


index
2024-03-22    8.360741
2024-03-25    8.563601
2024-03-26    8.760027
2024-03-27    8.842813
2024-03-28    9.096310
Freq: B, Name: price, dtype: float64

In [ ]:
from systems.forecast_scale_cap import ForecastScaleCap

my_config.instruments = ["SP500", "US10", "CORN", "SOFR"]
my_config.use_forecast_scale_estimates = True

fcs = ForecastScaleCap()
my_system = System([fcs, my_rules], data, config=my_config)

my_system.forecastScaleCap.get_forecast_scalar("SP500", "ewmac32").tail(5)


2026-08-27 16:45:08 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-08-27 16:45:08 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-08-27 16:45:08 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-08-27 16:45:08 DEBUG base_system {'stage': 'forecastScaleCap'} Getting cross sectional forecasts for scalar calculation for ewmac32 over CORN, SOFR, SP500, US10
2026-08-27 16:45:08 DEBUG base_system {'stage': 'rules', 'instrument_code': 'CORN'} Calculating raw forecast CORN for ewmac32
2026-08-27 16:45:08 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac32
2026-08-27 16:45:08 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SP500'} Calculating raw forecast SP500 for ewmac32
2026-08-27 16:45:08 DEBUG base_system {'stage': 'rules', 'instrument_code': 'US10'} Cal

index
2024-03-22    2.975047
2024-03-25    2.975032
2024-03-26    2.975035
2024-03-27    2.975045
2024-03-28    2.975110
Freq: B, dtype: float64

In [ ]:
my_config.forecast_scalars = dict(ewmac8=5.3, ewmac32=2.65)
my_config.use_forecast_scale_estimates = False

my_system = System([fcs, my_rules], data, config=my_config)

my_system.forecastScaleCap.get_forecast_scalar("SP500", "ewmac32").tail(5)

my_system.forecastScaleCap.get_capped_forecast("SP500", "ewmac32")


2026-08-27 16:47:46 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-08-27 16:47:46 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-08-27 16:47:46 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-08-27 16:47:46 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SP500'} Calculating raw forecast SP500 for ewmac32


2026-08-27 16:47:46 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SP500'} Calculating capped forecast for SP500 ewmac32


index
1982-09-14     NaN
1982-09-15     NaN
1982-09-16     NaN
1982-09-17     NaN
1982-09-20     NaN
              ... 
2024-03-22    20.0
2024-03-25    20.0
2024-03-26    20.0
2024-03-27    20.0
2024-03-28    20.0
Freq: B, Length: 10838, dtype: float64

In [ ]:
from systems.forecast_combine import ForecastCombine

combiner = ForecastCombine()
my_system = System([fcs, empty_rules, combiner], data, config=my_config)
my_system.combForecast.get_forecast_weights("SP500").tail(5)
my_system.combForecast.get_forecast_diversification_multiplier("SOFR").tail(5)


2026-08-27 16:55:54 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-08-27 16:55:54 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-08-27 16:55:54 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-08-27 16:55:54 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500'} Calculating forecast weights for SP500
2026-08-27 16:55:54 WARNING base_system {'stage': 'combForecast', 'instrument_code': 'SP500'} WARNING: No forecast weights  - using equal weights of 0.500 over all 2 trading rules in system
2026-08-27 16:55:54 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SP500'} Calculating capped forecast for SP500 ewmac32
2026-08-27 16:55:54 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SP500'} Calculating raw forecast SP500 for ewmac32
2026-08-27 16:55:54 DEBUG base_system {'st

index
2024-03-22    1.0
2024-03-25    1.0
2024-03-26    1.0
2024-03-27    1.0
2024-03-28    1.0
Freq: B, dtype: float64

In [ ]:
from systems.rawdata import RawData
from systems.positionsizing import PositionSizing
from systems.accounts.accounts_stage import Account

combiner = ForecastCombine()
raw_data = RawData()
position_size = PositionSizing()
my_account = Account()

my_config.forecast_weight_estimate = dict(method="one_period")
my_config.use_forecast_weight_estimates = True
my_config.use_forecast_div_mult_estimates = True

my_system = System(
    [my_account, fcs, my_rules, combiner, position_size, raw_data],
    data,
    config=my_config,
)

my_system.combForecast.get_forecast_weights("US10").tail(5)

my_system.combForecast.get_forecast_diversification_multiplier("US10").tail(5)


2026-08-27 17:02:32 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-08-27 17:02:32 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-08-27 17:02:32 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-08-27 17:02:32 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Calculating forecast weights for US10
2026-08-27 17:02:32 INFO base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Calculating raw forecast weights for US10
2026-08-27 17:02:32 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'CORN'} Calculating capped forecast for CORN ewmac32
2026-08-27 17:02:32 DEBUG base_system {'stage': 'rules', 'instrument_code': 'CORN'} Calculating raw forecast CORN for ewmac32
2026-08-27 17:02:33 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SOFR'} Calculating

index
2024-03-22    1.037744
2024-03-25    1.037740
2024-03-26    1.037735
2024-03-27    1.037731
2024-03-28    1.037727
Freq: B, dtype: float64

In [ ]:
my_config.forecast_weights = dict(ewmac8=0.5, ewmac32=0.5)
my_config.forecast_div_multiplier = 1.1
my_config.use_forecast_weight_estimates = False
my_config.use_forecast_div_mult_estimates = False
my_system = System(
    [fcs, empty_rules, combiner, raw_data, position_size, my_account],
    data,
    config=my_config,
)
my_system.combForecast.get_combined_forecast("SP500").tail(5)


2026-08-27 17:04:36 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-08-27 17:04:36 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-08-27 17:04:36 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-08-27 17:04:36 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500'} Calculating combined forecast for SP500
2026-08-27 17:04:36 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SP500'} Calculating capped forecast for SP500 ewmac32
2026-08-27 17:04:36 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SP500'} Calculating raw forecast SP500 for ewmac32


2026-08-27 17:04:36 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SP500'} Calculating capped forecast for SP500 ewmac8
2026-08-27 17:04:36 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SP500'} Calculating raw forecast SP500 for ewmac8
2026-08-27 17:04:36 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'SP500'} Calculating forecast weights for SP500
2026-08-27 17:04:36 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'CORN'} Calculating capped forecast for CORN ewmac32
2026-08-27 17:04:36 DEBUG base_system {'stage': 'rules', 'instrument_code': 'CORN'} Calculating raw forecast CORN for ewmac32
2026-08-27 17:04:36 DEBUG base_system {'stage': 'forecastScaleCap', 'instrument_code': 'SOFR'} Calculating capped forecast for SOFR ewmac32
2026-08-27 17:04:36 DEBUG base_system {'stage': 'rules', 'instrument_code': 'SOFR'} Calculating raw forecast SOFR for ewmac32
2026-08-27 17:04:37 DEBUG base_system {'stage': 'forecastScaleCap', '

index
2024-03-22    19.331206
2024-03-25    19.329044
2024-03-26    19.139832
2024-03-27    19.337329
2024-03-28    19.491053
Freq: B, dtype: float64

In [ ]:
my_system.combForecast.get_combined_forecast("US10").tail(5)


2026-08-28 13:15:34 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Calculating combined forecast for US10
2026-08-28 13:15:34 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Calculating forecast weights for US10
2026-08-28 13:15:34 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'US10'} Calculating daily prices for US10
2026-08-28 13:15:34 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'US10'} Calculating daily prices for US10
2026-08-28 13:15:35 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'US10'} Calculating daily volatility for US10
2026-08-28 13:15:35 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'US10'} Calculating daily prices for US10
2026-08-28 13:15:35 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US10'} Only this set of rules ['ewmac32', 'ewmac8'] is cheap enough to trade for US10
2026-08-28 13:15:35 DEBUG base_system {'stage': 'combForecast', 'instrument_code': 'US1

index
2024-03-22   -3.805354
2024-03-25   -3.683699
2024-03-26   -3.489892
2024-03-27   -2.964913
2024-03-28   -2.785518
Freq: B, dtype: float64

In [ ]:
my_config.percentage_vol_target = 25
my_config.notional_trading_capital = 500000
my_config.base_currency = "USD"

my_system = System(
    [fcs, empty_rules, combiner, raw_data, position_size], data, config=my_config
)

my_system.positionSize.get_subsystem_position("SP500").tail(5)


2026-08-28 13:17:35 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-08-28 13:17:35 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-08-28 13:17:35 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-08-28 13:17:35 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SP500'} Calculating subsystem position for SP500
2026-08-28 13:17:35 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SP500'} Calculating volatility scalar for SP500
2026-08-28 13:17:35 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SP500'} Calculating instrument value vol for SP500
2026-08-28 13:17:35 DEBUG base_system {'stage': 'positionSize', 'instrument_code': 'SP500'} Calculating instrument currency vol for SP500
2026-08-28 13:17:35 DEBUG base_system {'stage': 'rawdata', 'instrument_code': 'SP500'} Calculat

index
2024-03-22     9.502566
2024-03-25     9.638111
2024-03-26     9.689499
2024-03-27     9.813900
2024-03-28    10.060406
Freq: B, dtype: float64

In [ ]:
from systems.provided.futures_chapter15.basesystem import futures_system
from sysdata.config.configdata import Config

config = Config()  # default config = no instrument-list restriction
system = futures_system(config=config)

tradeable = system.get_instrument_list()  # after all exclusions
bad = system.get_list_of_bad_markets()  # cost/liquidity fails
restricted = system.get_list_of_markets_with_trading_restrictions()  # regulatory
dupes = (
    system.get_list_of_duplicate_instruments_to_remove()
)  # e.g. SP500 vs SP500_micro


2026-08-28 13:55:16 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 
2026-08-28 13:55:16 DEBUG base_system Following instruments are marked as 'ignore_instruments': not included: ['EXAMPLE']
2026-08-28 13:55:16 DEBUG base_system Following instruments removed entirely from sim: ['Another_thing', 'EXAMPLE', 'bad_thing']
2026-08-28 13:55:16 DEBUG base_system Following instruments are marked as 'bad_markets':  ['BAD_EXAMPLE']
2026-08-28 13:55:16 DEBUG base_system Following instruments have restricted trading:  ['RESTRICTED_EXAMPLE'] 
2026-08-28 13:55:16 DEBUG base_system Following instruments are 'duplicate_markets' ['Another_thing', 'bad_thing'] 


In [ ]:
dupes


['Another_thing', 'bad_thing']